# 🤟 BISINDO CSLR — Backend Server (Google Colab + GPU)

Notebook ini menjalankan **backend FastAPI** pipeline CSLR BISINDO di Google Colab dengan GPU T4.

Setelah server berjalan, Anda akan mendapatkan **URL publik** (via ngrok) yang dapat digunakan sebagai target backend di frontend lokal.

---

### Prasyarat
- Runtime: **GPU** (Runtime → Change runtime type → T4 GPU)
- Akun [ngrok](https://ngrok.com) gratis (untuk mendapat authtoken)

---

### Urutan Eksekusi
Jalankan sel **secara berurutan** dari atas ke bawah.

## ✅ Step 0 — Verifikasi GPU

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU tidak terdeteksi. Pastikan runtime sudah diset ke GPU.")

## 📦 Step 1 — Clone Repository

In [ ]:
import os

REPO_DIR = "/content/bisindo-cslr"
REPO_URL = "https://github.com/MahardikaPratama/bisindo-cslr.git"

if os.path.exists(REPO_DIR):
    print("Repo sudah ada, skip clone.")
else:
    !git clone --recursive "{REPO_URL}" "{REPO_DIR}"
    print("✅ Clone selesai.")

%cd {REPO_DIR}
!ls -la

## 🐍 Step 2 — Install Dependencies

In [ ]:
# Install semua dependencies yang dibutuhkan
!pip install -q \
    mediapipe==0.10.14 \
    "opencv-python>=4.8.0" \
    "numpy>=1.24.0,<2.0.0" \
    "matplotlib>=3.8.0" \
    "fastapi>=0.111.0" \
    "uvicorn>=0.30.0" \
    "python-multipart>=0.0.9" \
    scipy \
    pyyaml \
    tqdm \
    gdown \
    pyngrok

print("✅ Dependencies terinstall.")

## 🔽 Step 3 — Download Model dari Google Drive

In [ ]:
import gdown
import os

MODEL_DIR  = "/content/bisindo-cslr/mslr_iccv2025/model"
MODEL_PATH = os.path.join(MODEL_DIR, "best_dev_01.30_epoch39_model.pt")
GDRIVE_ID  = "1Uw6nJnR74DtNp3xhGi5kCT702I8II_As"

os.makedirs(MODEL_DIR, exist_ok=True)

if os.path.exists(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"✅ Model sudah ada ({size_mb:.0f} MB), skip download.")
else:
    print("⬇️  Mengunduh model dari Google Drive (~680 MB)...")
    url = f"https://drive.google.com/uc?id={GDRIVE_ID}"
    gdown.download(url, MODEL_PATH, quiet=False)
    size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"✅ Download selesai ({size_mb:.0f} MB) → {MODEL_PATH}")

## 🔑 Step 4 — Konfigurasi ngrok Authtoken

Daftar akun gratis di [https://ngrok.com](https://ngrok.com) → **Your Authtoken** → copy dan paste di bawah.

In [ ]:
from pyngrok import ngrok

NGROK_TOKEN = "3DXuk4IKY5lCWVt2VZo5X5fLCER_6ALsKCfwqfR8dJn7VMcNE"

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok authtoken dikonfigurasi.")

## 🚀 Step 5 — Jalankan Backend Server

> **Catatan:** Sel ini berjalan secara **background**. Tunggu sampai pesan `✅ Server siap` muncul sebelum lanjut ke Step 6.

In [ ]:
import subprocess
import time
import requests
import os

os.chdir("/content/bisindo-cslr")

# Jalankan uvicorn di background
server_process = subprocess.Popen(
    [
        "uvicorn", "app:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--log-level", "info"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd="/content/bisindo-cslr"
)

print("⏳ Menunggu server startup...")
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("✅ Server siap di http://127.0.0.1:8000")
            break
    except Exception:
        pass
else:
    print("❌ Server gagal start dalam 30 detik. Cek log di bawah.")
    out, _ = server_process.communicate(timeout=2)
    print(out.decode())

## 🌐 Step 6 — Buat Tunnel ngrok (URL Publik)

In [ ]:
from pyngrok import ngrok

# Buat tunnel HTTP ke port 8000
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print("=" * 60)
print("✅ BACKEND SIAP!")
print("=" * 60)
print(f"\n🔗 Public URL  : {PUBLIC_URL}")
print(f"📡 Health check: {PUBLIC_URL}/health")
print(f"📖 API Docs    : {PUBLIC_URL}/docs")
print("\n" + "=" * 60)
print("📋 LANGKAH SELANJUTNYA:")
print("   Salin URL di atas, lalu update vite.config.ts di frontend:")
print("")
print("   proxy: {")
print(f"     '/api'    : {{ target: '{PUBLIC_URL}', changeOrigin: true }},")
print(f"     '/preview': {{ target: '{PUBLIC_URL}', changeOrigin: true }},")
print("   }")
print("=" * 60)

## 🔍 Step 7 — Verifikasi Endpoint (Opsional)

In [ ]:
import requests
import json

# Test /health
r = requests.get(f"{PUBLIC_URL}/health")
print(f"/health → {r.status_code}: {r.json()}")

print("\n✅ Backend berjalan dan dapat diakses dari luar Colab.")
print("\n⚠️  PENTING: Jangan tutup tab Colab ini selama frontend digunakan.")
print("   Sesi ngrok akan mati jika notebook di-disconnect.")

## 📊 Step 8 — Monitor Log Server (Opsional)

Jalankan sel ini untuk melihat log server secara real-time.

> Tekan **stop (■)** di sebelah sel untuk menghentikan streaming log.

In [ ]:
import sys

print("📋 Streaming server log (Ctrl+C atau stop untuk berhenti)...\n")
try:
    for line in iter(server_process.stdout.readline, b''):
        decoded = line.decode('utf-8', errors='replace').rstrip()
        print(decoded, flush=True)
except KeyboardInterrupt:
    print("\n⏹️  Log streaming dihentikan.")

## 🛑 Step 9 — Hentikan Server (Opsional)

Jalankan hanya jika ingin menghentikan server sebelum sesi Colab berakhir.

In [ ]:
from pyngrok import ngrok as _ngrok

# Tutup tunnel ngrok
_ngrok.kill()

# Hentikan uvicorn
if 'server_process' in dir():
    server_process.terminate()
    server_process.wait()

print("✅ Server dan tunnel ngrok dihentikan.")